# SAC v2.18-P3b — Deterministic Dev-Set Evaluation

**Identical** to v2.18-P3b in every respect (architecture, reward, hyperparameters, noise schedule) **except**:
- Eval env uses **deterministic DEV_YEARS {2002, 2004, 2013} × {70%, 85%, 100%}** scheduling
- Uses `FixedScheduleEvalCallback` (same protocol as all TD3 runs)
- Adds collapse guard + nonfinite guard (diagnostic only)
- Writes `manifest.json` at run start

**Purpose:** Fair comparison with TD3 v2.21c. The original v2.18-P3b used randomized eval on training years with DEV_YEARS={2002,2016,2023}. This re-run uses the same dev set and eval protocol as TD3.


In [ ]:
# Mount Drive (persistent storage).
from google.colab import drive; drive.mount('/content/drive')
import os; DRIVE_ROOT='/content/drive/MyDrive/thesis_v218_p3b_runs'; os.makedirs(DRIVE_ROOT,exist_ok=True)
print('Drive mounted:', DRIVE_ROOT)


In [ ]:
# Clone repo + install deps (SB3 pinned 2.6.0).
import subprocess, sys, os
WORK = '/content'
repo = os.path.join(WORK, 'thesis')
if os.path.exists(repo):
    subprocess.run(['rm', '-rf', repo], check=True)
subprocess.run(['git', 'clone', 'https://github.com/taratorbati/thesis.git', repo], check=True)
os.chdir(repo); sys.path.insert(0, repo)
subprocess.run(['pip', 'install', '--quiet',
                'stable-baselines3==2.6.0', 'gymnasium', 'wandb', 'pytest'], check=True)
import torch
print(f'PyTorch: {torch.__version__}  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# WandB + GPU.
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY']=userdata.get('WANDB_API_KEY'); print('OK WANDB key loaded.')
except Exception as e:
    try:
        import getpass; k=getpass.getpass('WANDB key (Enter to skip): ').strip()
        if k: os.environ['WANDB_API_KEY']=k; print('OK set.')
        else: print('Skipping WandB.')
    except Exception: print('Skipping WandB.')
import subprocess; print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or 'no GPU')


In [ ]:
# Write the devset trainer into the cloned repo.
# SKIP this cell if you already pushed train_v218_p3b_devset.py to the repo.
import os
dst = '/content/thesis/src/rl/train_v218_p3b_devset.py'
content = bytes.fromhex('...').decode('utf-8')
with open(dst, 'w', encoding='utf-8') as f:
    f.write(content)
print(f'Written {len(content)} chars to {dst}')
# Verify import
from src.rl.train_v218_p3b_devset import train_sac_v218_p3b_devset
print('Import OK')


In [ ]:
# Pre-flight: smoke tests + 1000-step pilot exercising the devset eval wiring.
import subprocess, sys
print('Smoke tests...')
assert subprocess.run([sys.executable,'-m','pytest','tests/test_rl_smoke.py','-v','--tb=short']).returncode==0, 'SMOKE FAILED'
print('\nFactorized-critic tests...')
assert subprocess.run([sys.executable,'-m','pytest','tests/test_factorized_critic.py','-v','--tb=short']).returncode==0, 'CRITIC TESTS FAILED'
print('\n1000-step pilot (devset eval, compressed noise schedule)...')
from src.rl.train_v218_p3b_devset import train_sac_v218_p3b_devset
_ = train_sac_v218_p3b_devset(seed=999, output_dir='/content/pilot', wandb_project=None,
                              total_timesteps=1000, explore_decay_steps=300,
                              reinject_start=500, reinject_end=800)
print('\nOK pre-flight passed. Proceed.')


In [ ]:
# Full 250k training (SAC v2.18-P3b with deterministic dev-set eval).
# ~30-55 min A100 / ~2-2.5 h T4. All hyperparams identical to v2.18-P3b.
SEED = 0    # CHANGE per session
from src.rl.train_v218_p3b_devset import train_sac_v218_p3b_devset
model = train_sac_v218_p3b_devset(
    seed=SEED,
    output_dir='/content/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    ent_coef=0.002,                 # v2.18-P3b (unchanged)
    explore_sigma_start=0.30, explore_sigma_floor=0.0,
    explore_sigma_reinject=0.15,    # late pulse peak (unchanged)
    explore_decay_steps=60_000,
    reinject_start=150_000, reinject_end=180_000,
)
print('Training complete.')


In [ ]:
# Persist to Drive (CRITICAL). Run name includes timestamp.
import shutil, os, datetime, glob
# Find the run dir (includes timestamp in name)
pattern = f'/content/thesis/results/rl/sac_v218_p3b_devset_seed{SEED}_*'
runs = sorted(glob.glob(pattern), key=os.path.getmtime, reverse=True)
assert runs, f'No run dir matching {pattern}'
src = runs[0]
dst = os.path.join(DRIVE_ROOT, os.path.basename(src))
shutil.copytree(src, dst, ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
print('Archived to Drive:', dst)


In [ ]:
# Post-training 9-cell eval.
import subprocess, sys, os, glob
# Find the run dir
pattern = f'/content/thesis/results/rl/sac_v218_p3b_devset_seed{SEED}_*'
runs = sorted(glob.glob(pattern), key=os.path.getmtime, reverse=True)
assert runs, f'No run dir matching {pattern}'
run_dir = runs[0]
model_path = os.path.join(run_dir, 'best_model', 'best_model.zip')
run_name = os.path.basename(run_dir)
final_path = os.path.join(run_dir, f'{run_name}_final.zip')

print(f'Evaluating BEST checkpoint: {model_path}')
r = subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl',
    '--mode','eval','--model',model_path,
    '--scenario','all','--budget','all','--forecast','perfect'],
    capture_output=True, text=True)
print(r.stdout[-1500:])
if r.returncode!=0: print('STDERR:', r.stderr[-2000:])
assert r.returncode==0, 'PERFECT EVAL FAILED'

print('\nEvaluating BEST checkpoint (noisy forecast)...')
subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl',
    '--mode','eval','--model',model_path,
    '--scenario','all','--budget','all','--forecast','noisy','--noise-seed','42'],
    capture_output=False)

if os.path.exists(final_path):
    print('\nEvaluating FINAL checkpoint (perfect)...')
    subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl',
        '--mode','eval','--model',final_path,
        '--scenario','all','--budget','all','--forecast','perfect'],
        capture_output=False)

# Copy eval outputs to Drive.
import shutil, datetime
dst=os.path.join(DRIVE_ROOT, 'eval_sac_devset_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
shutil.copytree('/content/thesis/results/runs', dst, dirs_exist_ok=True)
print('Eval outputs archived to:', dst)


In [ ]:
# COMPARATIVE DIAGNOSTIC: SAC v2.18-devset vs TD3 v2.21c vs MPC
import pandas as pd, numpy as np, json, glob, os

# Auto-locate eval output
cands = [d for d in glob.glob('/content/thesis/results/runs/*v218*devset*') if os.path.isdir(d)]
cands += [d for d in glob.glob('/content/thesis/results/runs/*') if os.path.isdir(d) and
          glob.glob(os.path.join(d,'sac_perfect_det_wet*100*seed0.parquet'))]
OUTPUT_DIR = next(iter(sorted(set(cands), key=os.path.getmtime, reverse=True)), None)
print('SAC eval dir:', OUTPUT_DIR)

scenarios = ['dry_rice_100pct','dry_rice_85pct','dry_rice_70pct',
             'moderate_rice_100pct','moderate_rice_85pct','moderate_rice_70pct',
             'wet_rice_100pct','wet_rice_85pct','wet_rice_70pct']

def load_9cell(base, seed=0, prefix='sac_perfect_det_'):
    yields, droughts, wlogs = [], [], []
    for sc in scenarios:
        p = os.path.join(base, f'{prefix}{sc}_seed{seed}.json')
        if not os.path.exists(p): continue
        m = json.load(open(p)).get('final_metrics', json.load(open(p)))
        yields.append(m.get('yield_kg_ha', m.get('mean_yield_kg_ha', 0)))
        droughts.append(m.get('drought_days_per_agent', m.get('mean_drought_days_per_agent', 0)))
        wlogs.append(m.get('waterlog_days_per_agent', m.get('mean_waterlog_days_per_agent', 0)))
    return np.mean(yields), np.mean(droughts), np.mean(wlogs)

if OUTPUT_DIR:
    sy, sd, sw = load_9cell(OUTPUT_DIR)
    # MPC reference
    my, md, mw = 3810, 16.0, 7.7  # from committed data
    # TD3 v2.21c reference (3-seed mean)
    ty, td_, tw = 3800, 35.2, 6.5
    print('='*70)
    print(f'{"Metric":<22s}{"MPC":>8s}{"SAC devset":>12s}{"TD3 v2.21c":>12s}')
    print('-'*70)
    print(f'{"Yield (kg/ha)":<22s}{my:>8.0f}{sy:>12.0f}{ty:>12.0f}')
    print(f'{"% MPC":<22s}{"100%":>8s}{100*sy/my:>11.1f}%{100*ty/my:>11.1f}%')
    print(f'{"Drought-days":<22s}{md:>8.1f}{sd:>12.1f}{td_:>12.1f}')
    print(f'{"Waterlog-days":<22s}{mw:>8.1f}{sw:>12.1f}{tw:>12.1f}')
    print('='*70)


In [ ]:
# STABILITY + COVERAGE DIAGNOSTIC
import pandas as pd, os, glob
pattern = f'/content/thesis/results/rl/sac_v218_p3b_devset_seed{SEED}_*'
runs = sorted(glob.glob(pattern), key=os.path.getmtime, reverse=True)
run_dir = runs[0] if runs else None

if run_dir:
    # Bias ratio
    br = os.path.join(run_dir, 'bias_ratio_log.csv')
    if os.path.exists(br):
        b = pd.read_csv(br)
        print('Bias ratio trajectory:')
        print(b.to_string(index=False))
        neg = (b['q_pred_mean']<0).any()
        print(f'\n  q_pred_mean ever negative: {neg}')
    
    # Collapse guard
    cg = os.path.join(run_dir, 'collapse_guard_log.csv')
    if os.path.exists(cg):
        c = pd.read_csv(cg)
        print(f'\n  Collapse guard: max_frac_low={c["frac_low_rolling"].max():.3f}, '
              f'any_collapsed={c["collapsed"].any()}')
    
    # Coverage
    cov = os.path.join(run_dir, 'low_action_coverage_log.csv')
    if os.path.exists(cov):
        cv = pd.read_csv(cov)
        print(f'\n  Coverage: frac_low peak={cv["frac_low_action"].max():.3f}')
else:
    print(f'No run dir found matching {pattern}')


In [ ]:
# Resume from checkpoint (if session died).
# SEED = 0
# Find the latest run dir on Drive:
# import glob; print(sorted(glob.glob(f'{DRIVE_ROOT}/sac_v218_p3b_devset_seed{SEED}_*')))
# CKPT = '<Drive-path>/checkpoints/<run_name>_<STEP>_steps.zip'
# from src.rl.train_v212 import AsymmetricLRSAC
# from src.rl.networks import V216CTDESACPolicy
# model = AsymmetricLRSAC.load(CKPT, custom_objects={'policy_class': V216CTDESACPolicy})
# # model.learn(total_timesteps=..., reset_num_timesteps=False)
